## 실행 코드

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

code_dir = Path.cwd()
if (code_dir / "final_code" / "cd_aware_ad").is_dir():
    code_dir /= "final_code/cd_aware_ad"
elif (code_dir / "cd_aware_ad").is_dir():
    code_dir /= "cd_aware_ad"
sys.path.insert(0, str(code_dir))

from concept_drift_anomaly_detection import (
    compute_metrics,
    precompute_for_tuning,
    run_pipeline_with_params,
    visualize_cd_ad,
)
from tune_cd_ad import RESULTS_DIR, tune


def parameter_csv(mode):
    folder = RESULTS_DIR / "concept_drift"
    current = folder / f"cd_ad_best_{mode}_hyperparam_tuning_results2.csv"
    if current.is_file():
        return current
    previous_mode = "finetuned" if mode == "full_finetuned" else mode
    return folder / f"cd_ad_best_{previous_mode}_hyperparam_tuning_results copy.csv"


WINDOW = 64
AD_CONSECUTIVE = {"instop": 5, "grabbing": 3, "hitting": 1}
ACTUAL_COLS = ["accel_x", "accel_y", "accel_z", "gyro_x", "gyro_y", "gyro_z", "mag_x", "mag_y", "mag_z"]
PRED_COLS = [f"{column}_pred" for column in ACTUAL_COLS]


def rerun_detection(mode, model, abnormal, row, *, cd_pad_post, with_cd_label=False):
    suffix = "_with_cd_label" if with_cd_label else ""
    path = (
        RESULTS_DIR / "inference_df" / mode / f"{abnormal}_all_scenario"
        / f"{model}_3st_output{suffix}.csv"
    )
    df, errors, features, count = precompute_for_tuning(
        pd.read_csv(path), ACTUAL_COLS, PRED_COLS, WINDOW, 50, clip_percentile=95
    )
    return run_pipeline_with_params(
        df=df, err=errors, feats=features, T=count,
        window=WINDOW, n_baseline=50, consec_k=10,
        r=float(row["r"]), k_high=float(row["k_high"]), k_low=float(row["k_low"]),
        ad_ratio_th=float(row["ad_ratio_th"]), abnomal=abnormal,
        ad_consec_n_default=2, ad_consec_map=AD_CONSECUTIVE,
        cd_pad_pre=10, cd_pad_post=cd_pad_post, skip_ad_in_cd=True,
    )

### Hyperparameter Tuning Code

In [ ]:
tune(cd_pad_pre=10, r_list=(5, 7, 9))

### Hyperparameter 불러온 후 재실행

In [ ]:
if __name__ == "__main__":
    window = WINDOW
    skip_ad_in_cd = True

    out_fig_root = RESULTS_DIR / "figures"

    rerun_results = {}
    for mode in ["full_finetuned", "zeroshot"]:
        results = []
        param_csv = parameter_csv(mode)
        params_df = pd.read_csv(param_csv)

        for _, row in tqdm(params_df.iterrows(), total=len(params_df), desc=f"mode={mode}"):
            model  = row["model"]
            abnor  = row["abnomal"]
            kh     = float(row["k_high"])
            kl     = float(row["k_low"])
            ratio  = float(row["ad_ratio_th"])
            r      = float(row["r"])
            cd_pad_pre_used  = 10
            cd_pad_post_used = int(row["cd_pad_post"])

            df_out, df_win, baseline_segs, _, tau_low_pt, tau_high_pt = rerun_detection(
                mode, model, abnor, row, cd_pad_post=cd_pad_post_used,
            )

            y_true = df_out["label"].to_numpy().astype(int)
            y_pred = df_out["ad_flag"].to_numpy().astype(int)
            m      = compute_metrics(y_true, y_pred)

            # 저장된 메트릭과 즉시 비교 (디버깅용)
            saved_f1 = float(row["f1"]) if "f1" in params_df.columns else None
            saved_comparison = (
                f" | saved={saved_f1:.6f}  Δf1={m['f1'] - saved_f1:+.6f}"
                if saved_f1 is not None else ""
            )
            print(
                f"  [{mode}|{model}|{abnor}] "
                f"F1={m['f1']:.6f}{saved_comparison} | "
                f"Prec={m['precision']:.4f}, Rec={m['recall']:.4f}, PA-F1={m['point_adjust_f1']:.4f}"
            )

            result_row = {
                "mode": mode, "model": model, "abnomal": abnor,
                "k_high": kh, "k_low": kl, "ad_ratio_th": ratio, "r": r,
                "cd_pad_pre": cd_pad_pre_used, "cd_pad_post": cd_pad_post_used,
                "skip_ad_in_cd": skip_ad_in_cd,
                **m,
            }
            results.append(result_row)

            fig_path = f"{out_fig_root}/{mode}/{model}_{abnor}_rerun.png"
            title_prefix = (
                f"[{mode}|{model}|{abnor}] "
                f"kh={kh}, kl={kl}, ratio={ratio}, r={r} | "
                f"F1={m['f1']:.4f}, PA-F1={m['point_adjust_f1']:.4f}"
            )
            visualize_cd_ad(
                df=df_out, df_win=df_win, baseline_segments=baseline_segs,
                tau_low_pt=tau_low_pt, tau_high_pt=tau_high_pt,
                window=window, title_prefix=title_prefix, save=fig_path, titled=True,
            )

        rerun_results[mode] = pd.DataFrame(results)

### CD와 함께 평가

In [ ]:
# -------------------------
# CD 성능 평가 (window-level cd_label → point-level cd_flag 비교)
# -------------------------
def compute_cd_metrics(df_out, df_win_out, window):
    """
    cd_label  : df_out의 point-level GT (새로 추가된 컬럼)
    cd_flag   : pipeline이 예측한 point-level CD flag

    df_win_out의 window-level cd_flag를 point-level로 펼쳐서
    df_out의 cd_label과 비교합니다.
    """
    N = len(df_out)

    # point-level GT
    if "cd_label" not in df_out.columns:
        raise ValueError("df_out에 'cd_label' 컬럼이 없습니다.")

    y_true_cd = pd.to_numeric(df_out["cd_label"], errors="coerce").fillna(0).astype(int).to_numpy()
    y_pred_cd = df_out["cd_flag"].astype(int).to_numpy()

    return compute_metrics(y_true_cd, y_pred_cd)


# -------------------------
# Visualization (single figure)
# -------------------------
def visualize_cd_ad_comparison(
    df, df_win, baseline_segments, cd_segments, tau_low_pt, tau_high_pt,
    window=64, title_prefix="", save=None,
    t_start=None, t_end=None,
):
    import matplotlib
    TITLE_FONTSIZE  = 23
    LEGEND_FONTSIZE = 15

    # ── 구간 슬라이싱 ───────────────────────────────────────────
    df          = df.copy()
    df_win      = df_win.copy()
    tau_low_pt  = tau_low_pt.copy()
    tau_high_pt = tau_high_pt.copy()

    def to_ts(s):
        if s is None:
            return None
        ts = pd.Timestamp(s)
        return ts.tz_localize("UTC") if ts.tzinfo is None else ts

    ts0 = to_ts(t_start)
    ts1 = to_ts(t_end)

    t_all     = pd.to_datetime(df["timestamp"],            utc=True)
    t_all_win = pd.to_datetime(df_win["timestamp_start"],  utc=True)

    pt_mask  = np.ones(len(df),     dtype=bool)
    win_mask = np.ones(len(df_win), dtype=bool)
    if ts0:
        pt_mask  &= (t_all     >= ts0).values
        win_mask &= (t_all_win >= ts0).values
    if ts1:
        pt_mask  &= (t_all     <= ts1).values
        win_mask &= (t_all_win <= ts1).values

    win_global_idx = np.where(win_mask)[0]
    df          = df[pt_mask].reset_index(drop=True)
    tau_low_pt  = tau_low_pt[pt_mask]
    tau_high_pt = tau_high_pt[pt_mask]
    df_win      = df_win[win_mask].reset_index(drop=True)

    # ── 공통 변수 ────────────────────────────────────────────────
    t       = df["timestamp"].to_numpy()
    ad_mask = df["ad_flag"].astype(bool).to_numpy()

    has_ad_gt = "label" in df.columns
    has_cd_gt = "cd_label" in df.columns

    if has_ad_gt:
        ad_gt_mask = pd.to_numeric(df["label"],    errors="coerce").fillna(0).astype(bool).to_numpy()
    if has_cd_gt:
        cd_gt_mask = pd.to_numeric(df["cd_label"], errors="coerce").fillna(0).astype(bool).to_numpy()

    actual_cols_ = ["accel_x", "accel_y", "accel_z", "gyro_x", "gyro_y", "gyro_z", "mag_x", "mag_y", "mag_z"]
    actual_cols_ = [c for c in actual_cols_ if c in df.columns]

    cmap   = matplotlib.colormaps["tab10"]
    colors = [cmap(i) for i in range(len(actual_cols_))]

    def shade(ax, x, mask, alpha, label, color):
        if not mask.any():
            return
        y0, y1 = ax.get_ylim()
        ax.fill_between(x, y0, y1, where=mask, alpha=alpha, label=label, color=color)
        ax.set_ylim(y0, y1)

    fig, axes = plt.subplots(3, 1, figsize=(22, 15), sharex=False)

    # ── ① Actual + CD GT + AD GT ────────────────────────────────
    ax = axes[0]
    for col, c in zip(actual_cols_, colors):
        ax.plot(t, df[col].to_numpy(), lw=0.9, label=col, color=c)
    ax.relim(); ax.autoscale_view()

    if has_ad_gt:
        shade(ax, t, ad_gt_mask, 0.25, "AD GT",  "black")
    if has_cd_gt:
        shade(ax, t, cd_gt_mask, 0.20, "CD GT",  "purple")

    ax.set_title("1. Actual Signals + Grond Truth",
                 fontsize=TITLE_FONTSIZE, fontweight="bold")

    handles, labels = ax.get_legend_handles_labels()
    sig_h = handles[:len(actual_cols_)];  sig_l = labels[:len(actual_cols_)]
    ext_h = handles[len(actual_cols_):];  ext_l = labels[len(actual_cols_):]
    leg1  = ax.legend(sig_h, sig_l, loc="upper left",  fontsize=LEGEND_FONTSIZE,
                      ncol=3, framealpha=0.85)
    ax.add_artist(leg1)
    if ext_h:
        ax.legend(ext_h, ext_l, loc="upper right", fontsize=LEGEND_FONTSIZE,
                  framealpha=0.85)

    # ── ② Concept Drift Score ────────────────────────────────────
    ax = axes[1]
    t_w  = df_win["timestamp_start"].to_numpy()
    cd_s = df_win["cd_score"].to_numpy()
    cd_t = df_win["cd_th"].to_numpy()

    # baseline / cd 구간 → 슬라이싱 후 로컬 인덱스로 변환
    base_mask_win = np.zeros(len(df_win), dtype=bool)
    for ws, we in baseline_segments:
        for li, gi in enumerate(win_global_idx):
            if ws <= gi < we:
                base_mask_win[li] = True

    cd_mask_win = np.zeros(len(df_win), dtype=bool)
    for ws, we in cd_segments:
        for li, gi in enumerate(win_global_idx):
            if ws <= gi < we:
                cd_mask_win[li] = True

    ax.plot(t_w, cd_s, lw=1.0,       label="Mahalanobis D", color="darkorchid")
    ax.plot(t_w, cd_t, "--", lw=1.2, label="CD threshold",  color="orange")
    ax.relim(); ax.autoscale_view()
    shade(ax, t_w, base_mask_win, 0.25, "Baseline",   "gold")
    shade(ax, t_w, cd_mask_win,   0.30, "CD flagged", "royalblue")

    ax.set_title("2. Concept Drift Score (Mahalanobis)",
                 fontsize=TITLE_FONTSIZE, fontweight="bold")
    ax.legend(loc="upper right", fontsize=LEGEND_FONTSIZE, framealpha=0.85)

    # ── ③ Error Score + Adaptive Threshold ──────────────────────
    ax = axes[2]
    ax.plot(t, df["error_score"].to_numpy(), lw=1.0, label="error_score", color="steelblue")
    ax.plot(t, tau_high_pt, "--", lw=1.3,            label="τ_high",      color="tomato")
    ax.plot(t, tau_low_pt,  "--", lw=1.3,            label="τ_low",       color="mediumseagreen")
    ax.relim(); ax.autoscale_view()
    shade(ax, t, ad_mask, 0.30, "AD (predicted)", "red")

    ax.set_title("3. Anomaly Score + Adaptive Thresholds",
                 fontsize=TITLE_FONTSIZE, fontweight="bold")
    ax.legend(loc="upper right", fontsize=LEGEND_FONTSIZE, framealpha=0.85)

    for ax in axes:
        ax.set_xlabel("timestamp", fontsize=LEGEND_FONTSIZE)
        ax.tick_params(labelsize=LEGEND_FONTSIZE - 1)

    fig.suptitle(title_prefix, fontsize=TITLE_FONTSIZE + 5,
                 fontweight="bold", y=1.005)
    plt.tight_layout()

    if save:
        os.makedirs(os.path.dirname(save) or ".", exist_ok=True)
        plt.savefig(save, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"[SAVED] {save}")
    else:
        plt.show()


# =========================================================
# Main: 저장된 하이퍼파라미터 로드 → 재실행 → CD/AD 동시 평가
# =========================================================
if __name__ == "__main__":
    window = WINDOW

    cd_pad_pre = 10
    cd_pad_post = 0
    skip_ad_in_cd = True

    # ── 경로 설정 ──────────────────────────────────────────
    out_csv_dir = RESULTS_DIR / "concept_drift"
    out_fig_root = RESULTS_DIR / "figures"

    # ── 저장된 하이퍼파라미터 로드 ─────────────────────
    param_csv = parameter_csv("zeroshot")
    params_df = pd.read_csv(param_csv)

    results = []

    for i in range(3):
        row = params_df.iloc[i]
        mode = row.get("mode", "zeroshot")
        model  = row["model"]
        abnor  = row["abnomal"]
        kh     = float(row["k_high"])
        kl     = float(row["k_low"])
        ratio  = float(row["ad_ratio_th"])
        r      = float(row["r"])
        ad_consec_n_used = int(row.get("ad_consec_n", AD_CONSECUTIVE.get(abnor, 2)))

        df_out, df_win, baseline_segments, cd_segments, tau_low_pt, tau_high_pt = rerun_detection(
            mode, model, abnor, row, cd_pad_post=cd_pad_post, with_cd_label=True,
        )

        # ── AD 성능 ────────────────────────────────────
        y_true_ad = df_out["label"].to_numpy().astype(int)
        y_pred_ad = df_out["ad_flag"].to_numpy().astype(int)
        ad_metrics = compute_metrics(y_true_ad, y_pred_ad)

        # ── CD 성능 ────────────────────────────────────
        cd_metrics = compute_cd_metrics(df_out, df_win, window)

        # ── 결과 기록 ──────────────────────────────────
        result_row = {
            "mode":          mode,
            "model":         model,
            "abnomal":       abnor,
            "k_high":        kh,
            "k_low":         kl,
            "ad_ratio_th":   ratio,
            "ad_consec_n":   ad_consec_n_used,
            "cd_pad_pre":    int(cd_pad_pre),
            "cd_pad_post":   int(cd_pad_post),
            "skip_ad_in_cd": bool(skip_ad_in_cd),
            "r":             r,
            # AD metrics (접두사 ad_)
            **{f"ad_{k}": v for k, v in ad_metrics.items()},
            # CD metrics (접두사 cd_)
            **{f"cd_{k}": v for k, v in cd_metrics.items()},
        }
        results.append(result_row)

        print(
            f"  [{mode}|{model}|{abnor}] "
            f"AD  → F1={ad_metrics['f1']:.4f}, PA-F1={ad_metrics['point_adjust_f1']:.4f}, "
            f"Prec={ad_metrics['precision']:.4f}, Rec={ad_metrics['recall']:.4f} | "
            f"CD  → F1={cd_metrics['f1']:.4f}, PA-F1={cd_metrics['point_adjust_f1']:.4f}, "
            f"Acc={cd_metrics['accuracy']:.4f}, Bal.acc={cd_metrics['balanced_accuracy']:.4f}"
            f"Prec={cd_metrics['precision']:.4f}, Rec={cd_metrics['recall']:.4f}"
        )

        # ── 시각화 저장 ────────────────────────────────
        title_prefix = (
            f"[{mode} | {model} | {abnor}] "
            f"kh={kh:.2f}, kl={kl:.2f}, ratio={ratio:.2f}, r={r:.2f}, "
            f"ad_consec_n={ad_consec_n_used}, cd_pad_pre={cd_pad_pre} | "
            f"AD F1={ad_metrics['f1']:.4f} | CD F1={cd_metrics['f1']:.4f}"
        )
        fig_path = f"{out_fig_root}/{mode}/{model}_{abnor}_eval.png"
        visualize_cd_ad_comparison(
            df=df_out,
            df_win=df_win,
            baseline_segments=baseline_segments,  # baseline_segs → baseline_segments
            cd_segments=cd_segments,              # cd_segs → cd_segments
            tau_low_pt=tau_low_pt,
            tau_high_pt=tau_high_pt,
            # title_prefix=title_prefix,
            t_start="2025-11-10 12:52:00",
            t_end="2025-11-10 13:03:00"
        )

        # ── CSV 저장 ───────────────────────────────────────
        os.makedirs(out_csv_dir, exist_ok=True)
        out_df = pd.DataFrame(results)
        out_csv = f"{out_csv_dir}/eval_ad_cd_{mode}.csv"
        out_df.to_csv(out_csv, index=False)
        print(f"\n[SAVED] {out_csv}")